# 🚀 업비트 자동 매매 시스템 v4.0

## BTC·SOL·ETH·XRP | 1시간봉 | LightGBM·XGBoost·GRU·앙상블 | Optuna | 리스크관리 v4

### v4 핵심 개선사항 (v3 대비)
- ✅ **타임프레임 상향**: 5분봉 → **1시간봉** (수수료·슬리피지 대비 신호 품질 향상)
- ✅ **예측 horizon 상향**: 6봉(30분) → **12봉(12시간)** (충분한 수익 여지 확보)
- ✅ **레이블 기준 강화**: ±0.5% → **±1.5%** (횡보 구간 필터, 노이즈 감소)
- ✅ **학습률 스케줄러**: CosineAnnealing + WarmUp (DL 모델 수렴 품질 개선)
- ✅ **클래스 불균형 보정**: `class_weight` + focal loss (횡보 편향 제거)
- ✅ **피처 정규화 수정**: 테스트 셋 leakage 제거 (train fit → test transform only)
- ✅ **Walk-Forward 실구현**: expanding window, 6-fold
- ✅ **백테스트 버그 수정**: exec_price 경계, hold_bar 이중관리, 피라미딩 선언 불일치
- ✅ **AutoTrader 클래스명 통일**: AutoTraderV3
- ✅ **손절 최소 보유 분리**: 손절은 3봉 이후, 신호 매도는 MIN_HOLD_BARS 유지


In [ ]:
# ── 필수 라이브러리 설치 (최초 1회) ──
import subprocess, sys

pkgs = [
    'pyupbit', 'ta', 'lightgbm', 'xgboost', 'optuna',
    'tensorflow', 'scikit-learn', 'pandas', 'numpy',
    'joblib', 'flask', 'requests', 'matplotlib',
]
for p in pkgs:
    try:
        __import__(p.replace('-','_').split('>=')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p, '-q'])

print('✅ 라이브러리 준비 완료')


In [ ]:
import os, json, time, datetime, warnings, logging
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import pyupbit

from sklearn.preprocessing      import RobustScaler
from sklearn.model_selection    import TimeSeriesSplit
from sklearn.metrics            import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import lightgbm  as lgb
import xgboost   as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import tensorflow as tf
from tensorflow.keras import layers, callbacks, optimizers, losses
from tensorflow.keras import mixed_precision
import ta

warnings.filterwarnings('ignore')
logging.getLogger('lightgbm').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')

# GPU-only training. Stop immediately if CUDA GPU is unavailable.
GPU_ONLY = True
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
GPU_DEVICES = tf.config.list_physical_devices('GPU')
if GPU_ONLY and not GPU_DEVICES:
    raise RuntimeError('GPU not detected. Check RTX 4090 / CUDA / cuDNN before running.')
for gpu in GPU_DEVICES:
    tf.config.experimental.set_memory_growth(gpu, True)
TF_DEVICE = '/GPU:0'
mixed_precision.set_global_policy('mixed_float16')
XGB_GPU_PARAMS = {'tree_method': 'hist', 'device': 'cuda'}
LGBM_GPU_PARAMS = {'device_type': 'gpu', 'gpu_platform_id': 0, 'gpu_device_id': 0}
print(f'GPU device: {GPU_DEVICES[0].name if GPU_DEVICES else "NONE"}')

print(f'TensorFlow {tf.__version__}  |  LightGBM {lgb.__version__}  |  XGBoost {xgb.__version__}')


In [ ]:
# ????????????????????????????????????????????????????????????????
#  ??
# ????????????????????????????????????????????????????????????????

# Paper trading only. Real orders and Telegram are disabled.
UPBIT_ACCESS_KEY = 'YOUR_ACCESS_KEY'
UPBIT_SECRET_KEY = 'YOUR_SECRET_KEY'
TELEGRAM_TOKEN   = None
TELEGRAM_CHAT_ID = None
PAPER_TRADING_ONLY = True

INITIAL_CAPITAL  = 1_000_000
MIN_ORDER        = 5_000

STOP_LOSS_PCT    = 0.04
TAKE_PROFIT_PCT  = 0.12
DAILY_LOSS_LIMIT = 0.06
KELLY_FRACTION   = 0.20
SLIPPAGE_RATE    = 0.0005
FEE_RATE         = 0.0005
COST_BUFFER_PCT  = (FEE_RATE + SLIPPAGE_RATE) * 2
VOL_LABEL_MULT   = 0.80
MAX_LABEL_THRESHOLD = 0.025
RECENCY_HALFLIFE_DAYS = 21
SIGNAL_EMA_ALPHA = 0.35

MIN_HOLD_BARS    = 12        # 5m x 12 = 1 hour
STOP_MIN_BARS    = 6         # 5m x 6 = 30 minutes
KELLY_MIN_TRADES = 20
KELLY_FIXED_FRAC = 0.03
PROB_THRESHOLD   = 0.15
MODEL_CONF_MIN   = 0.45

TICKERS = {
    'BTC': 'KRW-BTC',
    'SOL': 'KRW-SOL',
    'ETH': 'KRW-ETH',
    'XRP': 'KRW-XRP',
}
TIMEFRAMES = {
    '5m':  'minute5',
    '15m': 'minute15',
    '1h':  'minute60',
}
BASE_TIMEFRAME   = '5m'
CANDLE_INTERVAL  = TIMEFRAMES[BASE_TIMEFRAME]
CANDLE_COUNT     = None       # None = collect as much historical data as Upbit returns
LIVE_REFRESH_BARS = 600
SEQUENCE_LEN     = 96         # 8 hours of 5m bars
PREDICT_HORIZON  = 12         # predict 1 hour ahead
LABEL_THRESHOLD  = 0.006      # +/-0.6% over horizon
TRAIN_RATIO      = 0.70
VAL_RATIO        = 0.15
WF_FOLDS         = 6
RANDOM_STATE     = 42
OPTUNA_TRIALS    = 50
FAST_OPTUNA_TRIALS = 12
CONTINUOUS_LEARNING = True
RETRAIN_EVERY_CYCLES = 12     # LOOP_SEC=300 => roughly hourly
SAVE_DATA_EVERY_CYCLE = True
DATA_DIR = Path('data')
BARS_PER_DAY = 24 * 12

DL_NAMES = ['GRU', 'LSTM', 'Transformer']

for d in ['models', 'models/final', 'logs', 'data', 'reports']:
    os.makedirs(d, exist_ok=True)

upbit = None
print('Paper trading only: real Upbit orders and Telegram are disabled.')
print(f'\n[v4 paper parameters]')
print(f'  timeframes: {list(TIMEFRAMES.keys())} | base: {BASE_TIMEFRAME} | horizon: {PREDICT_HORIZON} bars')
print(f'  label threshold: +/-{LABEL_THRESHOLD*100:.2f}% | GPU only: {GPU_ONLY}')
print(f'  continuous learning: {CONTINUOUS_LEARNING} | retrain every {RETRAIN_EVERY_CYCLES} cycles')


In [ ]:
# ????????????????????????????????????????????????????????????????
#  ??? ?? (5m/15m/1h, max history + live cache)
# ????????????????????????????????????????????????????????????????

def _data_path(ticker: str, tf_name: str) -> Path:
    DATA_DIR.mkdir(exist_ok=True)
    return DATA_DIR / f'{ticker.replace("-", "_")}_{tf_name}.csv'

def _load_cached_ohlcv(ticker: str, tf_name: str) -> pd.DataFrame:
    path = _data_path(ticker, tf_name)
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df[~df.index.duplicated(keep='last')].sort_index()

def _save_cached_ohlcv(ticker: str, tf_name: str, df: pd.DataFrame):
    if df is None or df.empty:
        return
    df = df[~df.index.duplicated(keep='last')].sort_index()
    df.to_csv(_data_path(ticker, tf_name), encoding='utf-8-sig')

def fetch_ohlcv(ticker: str, interval: str = CANDLE_INTERVAL, count: int | None = CANDLE_COUNT) -> pd.DataFrame:
    dfs, to = [], None
    per_req, fetched = 200, 0
    last_oldest, empty_hits = None, 0
    while True:
        n = per_req if count is None else min(per_req, count - fetched)
        if n <= 0:
            break
        try:
            tmp = pyupbit.get_ohlcv(ticker, interval=interval, count=n, to=to)
        except Exception as e:
            print(f'  {ticker} {interval} fetch error: {e}')
            time.sleep(1.0)
            continue
        if tmp is None or tmp.empty:
            empty_hits += 1
            if empty_hits >= 2:
                break
            time.sleep(0.5)
            continue
        tmp = tmp[~tmp.index.duplicated(keep='last')].sort_index()
        oldest = tmp.index[0]
        if last_oldest is not None and oldest >= last_oldest:
            break
        dfs.append(tmp)
        fetched += len(tmp)
        last_oldest = oldest
        to = oldest.strftime('%Y-%m-%d %H:%M:%S')
        time.sleep(0.13)
        if len(tmp) < n:
            break
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs).sort_index()
    df = df[~df.index.duplicated(keep='last')]
    return df.tail(count) if count is not None else df

def update_ohlcv_cache(ticker: str, tf_name: str, full: bool = False) -> pd.DataFrame:
    cached = _load_cached_ohlcv(ticker, tf_name)
    interval = TIMEFRAMES[tf_name]
    fetch_count = None if (full or cached.empty) else LIVE_REFRESH_BARS
    fresh = fetch_ohlcv(ticker, interval=interval, count=fetch_count)
    if fresh.empty:
        return cached
    merged = pd.concat([cached, fresh]).sort_index() if not cached.empty else fresh
    merged = merged[~merged.index.duplicated(keep='last')]
    _save_cached_ohlcv(ticker, tf_name, merged)
    return merged

def refresh_all_market_data(full: bool = False):
    global raw_data
    raw_data = {}
    mode = 'max history' if full else 'latest refresh'
    print(f'Collecting market data ({mode}): 5m / 15m / 1h')
    for name, ticker in TICKERS.items():
        raw_data[ticker] = {}
        for tf_name in TIMEFRAMES:
            df = update_ohlcv_cache(ticker, tf_name, full=full)
            raw_data[ticker][tf_name] = df
            if df.empty:
                print(f'  {name} {tf_name}: empty')
            else:
                print(f'  {name} {tf_name}: {len(df):,} bars  {df.index[0]} ~ {df.index[-1]}')
    print('Data collection complete')
    return raw_data

raw_data = refresh_all_market_data(full=True)


In [ ]:
# Feature engineering (5m base + joined higher timeframe context)

def add_features_single(df: pd.DataFrame, prefix: str = '') -> pd.DataFrame:
    df = df.copy()
    c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']
    out = pd.DataFrame(index=df.index)
    eps = 1e-9

    out[f'{prefix}ret_1']  = c.pct_change(1)
    out[f'{prefix}ret_2']  = c.pct_change(2)
    out[f'{prefix}ret_3']  = c.pct_change(3)
    out[f'{prefix}ret_6']  = c.pct_change(6)
    out[f'{prefix}ret_12'] = c.pct_change(12)
    out[f'{prefix}ret_24'] = c.pct_change(24)
    out[f'{prefix}ret_48'] = c.pct_change(48)
    out[f'{prefix}ret_96'] = c.pct_change(96)
    out[f'{prefix}log_ret'] = np.log(c / c.shift(1))

    for n in [5, 10, 20, 50, 100]:
        ma = c.rolling(n).mean()
        ema = ta.trend.EMAIndicator(c, window=n).ema_indicator()
        out[f'{prefix}ma_{n}'] = ma
        out[f'{prefix}ema_{n}'] = ema
        out[f'{prefix}ma_ratio_{n}'] = c / (ma + eps) - 1
        out[f'{prefix}ema_ratio_{n}'] = c / (ema + eps) - 1
        out[f'{prefix}ma_slope_{n}'] = ma / (ma.shift(max(1, n // 2)) + eps) - 1
        change = c.diff(n).abs()
        path = c.diff().abs().rolling(n).sum()
        out[f'{prefix}eff_ratio_{n}'] = change / (path + eps)

    bb = ta.volatility.BollingerBands(c, window=20, window_dev=2)
    out[f'{prefix}bb_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / (bb.bollinger_mavg() + eps)
    out[f'{prefix}bb_pct']   = bb.bollinger_pband()
    out[f'{prefix}rsi_14']   = ta.momentum.RSIIndicator(c, window=14).rsi()
    out[f'{prefix}rsi_6']    = ta.momentum.RSIIndicator(c, window=6).rsi()
    out[f'{prefix}roc_12']   = ta.momentum.ROCIndicator(c, window=12).roc()
    out[f'{prefix}willr_14'] = ta.momentum.WilliamsRIndicator(h, l, c, lbp=14).williams_r()

    macd = ta.trend.MACD(c)
    out[f'{prefix}macd']        = macd.macd()
    out[f'{prefix}macd_signal'] = macd.macd_signal()
    out[f'{prefix}macd_diff']   = macd.macd_diff()
    sto = ta.momentum.StochasticOscillator(h, l, c)
    out[f'{prefix}stoch_k'] = sto.stoch()
    out[f'{prefix}stoch_d'] = sto.stoch_signal()

    atr = ta.volatility.AverageTrueRange(h, l, c, window=14).average_true_range()
    out[f'{prefix}atr_pct']       = atr / (c + eps)
    out[f'{prefix}volatility_12'] = out[f'{prefix}log_ret'].rolling(12).std()
    out[f'{prefix}volatility_24'] = out[f'{prefix}log_ret'].rolling(24).std()
    out[f'{prefix}volatility_48'] = out[f'{prefix}log_ret'].rolling(48).std()
    out[f'{prefix}ret_z_24']      = out[f'{prefix}log_ret'] / (out[f'{prefix}volatility_24'] + eps)

    vol_ma = v.rolling(24).mean()
    vol_sd = v.rolling(24).std()
    out[f'{prefix}vol_ratio'] = v / (vol_ma + eps)
    out[f'{prefix}vol_z_24']  = (v - vol_ma) / (vol_sd + eps)
    out[f'{prefix}vol_log']   = np.log1p(v)
    out[f'{prefix}obv_diff']  = ta.volume.OnBalanceVolumeIndicator(c, v).on_balance_volume().pct_change(5)
    out[f'{prefix}mfi_14']    = ta.volume.MFIIndicator(h, l, c, v, window=14).money_flow_index()
    out[f'{prefix}cmf_20']    = ta.volume.ChaikinMoneyFlowIndicator(h, l, c, v, window=20).chaikin_money_flow()

    spread = h - l
    body = c - o
    out[f'{prefix}hl_ratio']   = spread / (c + eps)
    out[f'{prefix}body_pct']   = body / (c + eps)
    out[f'{prefix}close_pos']  = (c - l) / (spread + eps)
    out[f'{prefix}upper_wick'] = (h - np.maximum(o, c)) / (spread + eps)
    out[f'{prefix}lower_wick'] = (np.minimum(o, c) - l) / (spread + eps)

    adx = ta.trend.ADXIndicator(h, l, c, window=14)
    out[f'{prefix}adx']     = adx.adx()
    out[f'{prefix}adx_pos'] = adx.adx_pos()
    out[f'{prefix}adx_neg'] = adx.adx_neg()
    out[f'{prefix}trend_strength'] = out[f'{prefix}adx'] * np.sign(out[f'{prefix}adx_pos'] - out[f'{prefix}adx_neg'])
    out[f'{prefix}cci_20'] = ta.trend.CCIIndicator(h, l, c, window=20).cci()
    return out

def add_features(data) -> pd.DataFrame:
    if not isinstance(data, dict):
        base_raw = data.copy()
        out = pd.concat([base_raw[['open','high','low','close','volume']], add_features_single(base_raw, prefix=f'{BASE_TIMEFRAME}_')], axis=1)
    else:
        base_raw = data[BASE_TIMEFRAME].copy()
        out = pd.concat([base_raw[['open','high','low','close','volume']], add_features_single(base_raw, prefix=f'{BASE_TIMEFRAME}_')], axis=1)
        for tf_name in [k for k in TIMEFRAMES if k != BASE_TIMEFRAME]:
            if tf_name not in data or data[tf_name].empty:
                continue
            tf_raw = data[tf_name].copy()
            tf_feat = add_features_single(tf_raw, prefix=f'{tf_name}_')
            aligned_close = tf_raw['close'].reindex(out.index, method='ffill')
            out = out.join(tf_feat.reindex(out.index, method='ffill'), how='left')
            out[f'{tf_name}_base_close_gap'] = base_raw['close'] / (aligned_close + 1e-9) - 1
    out['hour']     = out.index.hour
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    out['dow']      = out.index.dayofweek
    out['dow_sin']  = np.sin(2 * np.pi * out['dow'] / 7)
    out['dow_cos']  = np.cos(2 * np.pi * out['dow'] / 7)
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out.dropna(inplace=True)
    return out

def make_labels(df: pd.DataFrame, horizon: int = PREDICT_HORIZON, threshold: float = LABEL_THRESHOLD) -> pd.Series:
    close = df['close'].astype(float).values
    highs = np.column_stack([df['high'].shift(-i).values for i in range(1, horizon + 1)])
    lows  = np.column_stack([df['low'].shift(-i).values for i in range(1, horizon + 1)])
    future_close = df['close'].shift(-horizon)

    atr_col = f'{BASE_TIMEFRAME}_atr_pct'
    if atr_col in df.columns:
        vol = df[atr_col].rolling(12, min_periods=1).median()
    else:
        vol = df['close'].pct_change().rolling(horizon * 2, min_periods=3).std()
    dyn_thr = np.maximum(threshold, COST_BUFFER_PCT + vol.fillna(threshold).values * VOL_LABEL_MULT)
    dyn_thr = np.clip(dyn_thr, threshold, MAX_LABEL_THRESHOLD)

    up_hit = highs / (close[:, None] + 1e-9) - 1 >= dyn_thr[:, None]
    dn_hit = lows  / (close[:, None] + 1e-9) - 1 <= -dyn_thr[:, None]
    up_any = up_hit.any(axis=1)
    dn_any = dn_hit.any(axis=1)
    up_first = np.where(up_any, up_hit.argmax(axis=1) + 1, horizon + 1)
    dn_first = np.where(dn_any, dn_hit.argmax(axis=1) + 1, horizon + 1)
    close_ret = (future_close.values - close) / (close + 1e-9)

    label = np.ones(len(df), dtype=np.int32)
    label[up_first < dn_first] = 2
    label[dn_first < up_first] = 0
    tie = (up_first == dn_first) & (up_first <= horizon)
    label[tie & (close_ret > dyn_thr)] = 2
    label[tie & (close_ret < -dyn_thr)] = 0
    no_touch = (up_first > horizon) & (dn_first > horizon)
    label[no_touch & (close_ret > dyn_thr * 1.25)] = 2
    label[no_touch & (close_ret < -dyn_thr * 1.25)] = 0
    return pd.Series(label, index=df.index)

_sample_df = add_features(list(raw_data.values())[0]).head(1000)
FEATURE_COLS = [c for c in _sample_df.columns if c not in ['open','high','low','close','volume','label']]
print(f'Feature count: {len(FEATURE_COLS)}')
print(f'Base trading frame: {BASE_TIMEFRAME}; input frames: {list(TIMEFRAMES.keys())}')
print(f'Example features: {FEATURE_COLS[:10]} ...')


In [ ]:
# Dataset preparation with leakage control and recency weights

def make_sequences(X: np.ndarray, y: np.ndarray, seq_len: int = SEQUENCE_LEN):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i + seq_len])
        ys.append(y[i + seq_len])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int32)

def make_sample_weights(y: np.ndarray, cw_dict: dict, half_life_days: int = RECENCY_HALFLIFE_DAYS) -> np.ndarray:
    y = np.asarray(y, dtype=np.int32)
    n = len(y)
    half_life_bars = max(1, int(half_life_days * BARS_PER_DAY))
    age = np.arange(n - 1, -1, -1, dtype=np.float32)
    recency = np.power(0.5, age / half_life_bars)
    cls = np.array([cw_dict.get(int(v), 1.0) for v in y], dtype=np.float32)
    w = recency * cls
    return w / (np.mean(w) + 1e-9)

def prepare_datasets():
    global datasets
    datasets = {}
    for name, ticker in TICKERS.items():
        df = add_features(raw_data[ticker])
        lbl = make_labels(df)
        df = df.iloc[:-PREDICT_HORIZON]
        lbl = lbl.iloc[:-PREDICT_HORIZON]
        n = len(df)
        if n < SEQUENCE_LEN + 300:
            raise ValueError(f'{ticker}: not enough training data ({n})')
        n_tr = int(n * TRAIN_RATIO)
        n_val = int(n * VAL_RATIO)
        X_all = df[FEATURE_COLS].values.astype(np.float32)
        y_all = lbl.values.astype(np.int32)
        X_tr, y_tr = X_all[:n_tr], y_all[:n_tr]
        X_va, y_va = X_all[n_tr:n_tr+n_val], y_all[n_tr:n_tr+n_val]
        X_te, y_te = X_all[n_tr+n_val:], y_all[n_tr+n_val:]
        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr).astype(np.float32)
        X_va_s = scaler.transform(X_va).astype(np.float32)
        X_te_s = scaler.transform(X_te).astype(np.float32)
        classes = np.unique(y_tr)
        cw = compute_class_weight('balanced', classes=classes, y=y_tr)
        cw_dict = {int(c): float(w) for c, w in zip(classes, cw)}
        w_tr = make_sample_weights(y_tr, cw_dict)
        w_va = make_sample_weights(y_va, cw_dict)
        w_te = make_sample_weights(y_te, cw_dict)
        Xtr_seq, ytr_seq = make_sequences(X_tr_s, y_tr)
        Xva_seq, yva_seq = make_sequences(X_va_s, y_va)
        Xte_seq, yte_seq = make_sequences(X_te_s, y_te)
        datasets[ticker] = {
            'X_train': X_tr_s, 'y_train': y_tr,
            'X_val': X_va_s, 'y_val': y_va,
            'X_test': X_te_s, 'y_test': y_te,
            'Xtr_seq': Xtr_seq, 'ytr_seq': ytr_seq,
            'Xva_seq': Xva_seq, 'yva_seq': yva_seq,
            'Xte_seq': Xte_seq, 'yte_seq': yte_seq,
            'w_train': w_tr, 'w_val': w_va, 'w_test': w_te,
            'wtr_seq': w_tr[SEQUENCE_LEN:],
            'wva_seq': w_va[SEQUENCE_LEN:],
            'wte_seq': w_te[SEQUENCE_LEN:],
            'scaler': scaler, 'cw_dict': cw_dict,
            'test_df': df.iloc[n_tr+n_val:].copy(),
            'full_df': df, 'full_y': y_all,
        }
        lb = np.bincount(y_tr, minlength=3)
        print(f'  {name}: train={len(X_tr):,} val={len(X_va):,} test={len(X_te):,} labels=[down:{lb[0]} flat:{lb[1]} up:{lb[2]}]')
    print('Dataset ready')
    return datasets

datasets = prepare_datasets()


In [ ]:
# ════════════════════════════════════════════════════════════════
#  딥러닝 모델 빌더
#  ★ v4 개선: CosineAnnealing LR 스케줄러, Focal Loss, Dropout 강화
# ════════════════════════════════════════════════════════════════

N_FEATS = len(FEATURE_COLS)

def focal_loss(gamma: float = 2.0, alpha: float = 0.25):
    """Focal Loss — 클래스 불균형 대응."""
    def loss(y_true, y_pred):
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=3)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        cross_ent = -y_true_oh * tf.math.log(y_pred)
        prob_true = tf.reduce_sum(y_true_oh * y_pred, axis=-1, keepdims=True)
        weight    = alpha * tf.pow(1.0 - prob_true, gamma)
        return tf.squeeze(weight * tf.reduce_sum(cross_ent, axis=-1, keepdims=True), axis=-1)
    return loss


def cosine_warmup_schedule(epoch, warmup=5, total=50, lr_max=1e-3, lr_min=1e-5):
    if epoch < warmup:
        return lr_max * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total - warmup)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * progress))


def build_gru(seq_len=SEQUENCE_LEN, n_feats=N_FEATS,
              units=128, dropout=0.3, lr=1e-3):
    inp = layers.Input(shape=(seq_len, n_feats))
    x   = layers.GRU(units, return_sequences=True,
                     recurrent_dropout=0.0)(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.GRU(units // 2)(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    out = layers.Dense(3, activation='softmax', dtype='float32')(x)
    m   = tf.keras.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(lr),
              loss=focal_loss(), metrics=['accuracy'])
    return m


def build_lstm(seq_len=SEQUENCE_LEN, n_feats=N_FEATS,
               units=128, dropout=0.3, lr=1e-3):
    inp = layers.Input(shape=(seq_len, n_feats))
    x   = layers.Bidirectional(
              layers.LSTM(units, return_sequences=True,
                          recurrent_dropout=0.0))(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.LSTM(units // 2)(x)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(64, activation='relu')(x)
    x   = layers.BatchNormalization()(x)
    out = layers.Dense(3, activation='softmax', dtype='float32')(x)
    m   = tf.keras.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(lr),
              loss=focal_loss(), metrics=['accuracy'])
    return m


def build_transformer(seq_len=SEQUENCE_LEN, n_feats=N_FEATS,
                      d_model=64, n_heads=4, n_layers=2,
                      dropout=0.2, lr=1e-3):
    inp  = layers.Input(shape=(seq_len, n_feats))
    x    = layers.Dense(d_model)(inp)
    # Positional encoding
    pos  = tf.range(seq_len)
    pos  = tf.cast(pos, tf.float32)[:, tf.newaxis]
    div  = tf.exp(tf.range(0, d_model, 2, dtype=tf.float32)
                  * -(np.log(10000.0) / d_model))
    pe   = tf.concat([tf.sin(pos * div), tf.cos(pos * div)], axis=-1)
    x    = x + pe[tf.newaxis, :, :d_model]
    for _ in range(n_layers):
        # Multi-head attention
        attn = layers.MultiHeadAttention(num_heads=n_heads,
                                         key_dim=d_model // n_heads)(x, x)
        x    = layers.LayerNormalization()(x + layers.Dropout(dropout)(attn))
        # FFN
        ff   = layers.Dense(d_model * 4, activation='relu')(x)
        ff   = layers.Dense(d_model)(ff)
        x    = layers.LayerNormalization()(x + layers.Dropout(dropout)(ff))
    x    = layers.GlobalAveragePooling1D()(x)
    x    = layers.Dense(64, activation='relu')(x)
    x    = layers.Dropout(dropout)(x)
    out  = layers.Dense(3, activation='softmax', dtype='float32')(x)
    m    = tf.keras.Model(inp, out)
    m.compile(optimizer=optimizers.Adam(lr),
              loss=focal_loss(), metrics=['accuracy'])
    return m


def get_dl_callbacks(total_epochs=50, warmup=5,
                     lr_max=1e-3, lr_min=1e-5, patience=10):
    return [
        callbacks.LearningRateScheduler(
            lambda ep: cosine_warmup_schedule(ep, warmup, total_epochs, lr_max, lr_min),
            verbose=0),
        callbacks.EarlyStopping(monitor='val_loss', patience=patience,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=5, min_lr=lr_min, verbose=0),
    ]


print('✅ DL 모델 빌더 준비 완료')
print(f'  모델: GRU | Bi-LSTM | Transformer')
print(f'  Loss: Focal Loss(gamma=2)  |  LR: CosineAnnealing + WarmUp')


In [ ]:
# ML model training (LightGBM + XGBoost + Optuna, GPU only)

def tune_lgb(X_tr, y_tr, X_va, y_va, cw_dict, n_trials=OPTUNA_TRIALS, sw_tr=None, sw_va=None):
    def objective(trial):
        params = {
            'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
            'verbosity': -1, 'boosting_type': 'gbdt',
            'n_estimators': trial.suggest_int('n_estimators', 300, 1600),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'num_leaves': trial.suggest_int('num_leaves', 24, 192),
            'learning_rate': trial.suggest_float('learning_rate', 5e-4, 0.08, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 20, 180),
            'subsample': trial.suggest_float('subsample', 0.55, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.08),
            'max_bin': trial.suggest_categorical('max_bin', [63, 127, 255]),
            'random_state': RANDOM_STATE,
            **LGBM_GPU_PARAMS,
        }
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], eval_sample_weight=[sw_va] if sw_va is not None else None,
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        return f1_score(y_va, model.predict(X_va), average='macro', sample_weight=sw_va)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = lgb.LGBMClassifier(**best, objective='multiclass', num_class=3, verbosity=-1,
                                random_state=RANDOM_STATE, **LGBM_GPU_PARAMS)
    model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], eval_sample_weight=[sw_va] if sw_va is not None else None,
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return model, study.best_value

def tune_xgb(X_tr, y_tr, X_va, y_va, cw_dict, n_trials=OPTUNA_TRIALS, sw_tr=None, sw_va=None):
    if sw_tr is None:
        sw_tr = np.array([cw_dict.get(int(y), 1.0) for y in y_tr], dtype=np.float32)
    def objective(trial):
        params = {
            'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss',
            'verbosity': 0,
            'n_estimators': trial.suggest_int('n_estimators', 300, 1600),
            'max_depth': trial.suggest_int('max_depth', 3, 11),
            'learning_rate': trial.suggest_float('learning_rate', 5e-4, 0.08, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 40),
            'subsample': trial.suggest_float('subsample', 0.55, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 1.0),
            'gamma': trial.suggest_float('gamma', 0, 8),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 20.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 20.0, log=True),
            'max_bin': trial.suggest_categorical('max_bin', [128, 256, 512]),
            'random_state': RANDOM_STATE,
            **XGB_GPU_PARAMS,
        }
        model = xgb.XGBClassifier(**params)
        
        try:
            model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], sample_weight_eval_set=[sw_va] if sw_va is not None else None, verbose=False)
        except TypeError:
            model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
        return f1_score(y_va, model.predict(X_va), average='macro', sample_weight=sw_va)
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = study.best_params
    model = xgb.XGBClassifier(**best, objective='multi:softprob', num_class=3, verbosity=0,
                              random_state=RANDOM_STATE, **XGB_GPU_PARAMS)
    try:
        model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], sample_weight_eval_set=[sw_va] if sw_va is not None else None, verbose=False)
    except TypeError:
        model.fit(X_tr, y_tr, sample_weight=sw_tr, eval_set=[(X_va, y_va)], verbose=False)
    return model, study.best_value

print('ML tuners ready: LightGBM + XGBoost + Optuna + GPU + recency weights')


In [ ]:
# ════════════════════════════════════════════════════════════════
#  Walk-Forward Validation (실구현)
#  ★ v4: expanding window, 6-fold
# ════════════════════════════════════════════════════════════════

def walk_forward_eval(ticker: str, model_type: str = 'LightGBM',
                      n_folds: int = WF_FOLDS) -> dict:
    """
    Walk-Forward: 전체 데이터를 n_folds 구간으로 나눠
    각 fold에서 이전 모든 데이터로 학습 → 다음 구간 평가.
    """
    d    = datasets[ticker]
    X_s  = np.vstack([d['X_train'], d['X_val'], d['X_test']])
    y_s  = np.concatenate([d['y_train'], d['y_val'],  d['y_test']])
    n    = len(X_s)
    fold_size = n // (n_folds + 1)

    fold_results = []
    for fold in range(1, n_folds + 1):
        train_end = fold * fold_size
        test_end  = min((fold + 1) * fold_size, n)
        if test_end <= train_end:
            break
        X_tr_f, y_tr_f = X_s[:train_end],        y_s[:train_end]
        X_te_f, y_te_f = X_s[train_end:test_end], y_s[train_end:test_end]

        # ★ scaler를 fold train에만 fit
        sc = RobustScaler()
        X_tr_f = sc.fit_transform(X_tr_f)
        X_te_f = sc.transform(X_te_f)

        cw = compute_class_weight('balanced', classes=np.unique(y_tr_f), y=y_tr_f)
        cw_d = {int(c): float(w) for c, w in zip(np.unique(y_tr_f), cw)}

        if model_type == 'LightGBM':
            n_va = max(30, int(len(X_tr_f) * 0.15))
            mdl  = lgb.LGBMClassifier(
                n_estimators=300, max_depth=6, num_leaves=48,
                learning_rate=0.02, subsample=0.8, colsample_bytree=0.8,
                class_weight=cw_d, verbosity=-1, random_state=RANDOM_STATE, **LGBM_GPU_PARAMS)
            mdl.fit(X_tr_f, y_tr_f,
                    eval_set=[(X_te_f, y_te_f)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(-1)])
        elif model_type == 'XGBoost':
            sw = np.array([cw_d.get(int(y), 1.0) for y in y_tr_f])
            mdl = xgb.XGBClassifier(
                n_estimators=300, max_depth=5, learning_rate=0.02,
                subsample=0.8, colsample_bytree=0.8,
                objective='multi:softprob', num_class=3,
                verbosity=0, random_state=RANDOM_STATE, **XGB_GPU_PARAMS)
            mdl.fit(X_tr_f, y_tr_f, sample_weight=sw,
                    eval_set=[(X_te_f, y_te_f)], verbose=False)
        else:
            raise ValueError(f'Unknown model_type: {model_type}')

        pred = mdl.predict(X_te_f)
        acc  = accuracy_score(y_te_f, pred)
        f1   = f1_score(y_te_f, pred, average='macro')
        fold_results.append({'fold': fold, 'acc': acc, 'f1': f1,
                              'train_n': train_end, 'test_n': len(X_te_f)})

    df_r = pd.DataFrame(fold_results)
    return {
        'folds':   df_r,
        'mean_acc': df_r['acc'].mean(),
        'mean_f1':  df_r['f1'].mean(),
        'std_f1':   df_r['f1'].std(),
    }


print('✅ Walk-Forward 평가기 준비 완료')
print(f'  방식: Expanding Window  |  Folds: {WF_FOLDS}')

# Walk-Forward 실행 (LightGBM 기준)
print('\n  Walk-Forward 검증 실행 중...')
wf_results = {}
for name, ticker in TICKERS.items():
    r = walk_forward_eval(ticker, model_type='LightGBM')
    wf_results[ticker] = r
    print(f'  {name}: mean_acc={r["mean_acc"]:.4f}  '
          f'mean_f1={r["mean_f1"]:.4f}±{r["std_f1"]:.4f}')
print('\n✅ Walk-Forward 완료')


In [ ]:
# Full model training (weighted, GPU-only)

EPOCHS = 120
BATCH = 512

def train_all_models(optuna_trials=OPTUNA_TRIALS, dl_epochs=EPOCHS):
    global all_models, all_results
    all_models = {ticker: {} for ticker in TICKERS.values()}
    all_results = {ticker: {} for ticker in TICKERS.values()}
    for name, ticker in TICKERS.items():
        d, cwd = datasets[ticker], datasets[ticker]['cw_dict']
        print(f'\n{"="*60}\n  [{name}] training\n{"="*60}')
        lgb_mdl, lgb_f1 = tune_lgb(d['X_train'], d['y_train'], d['X_val'], d['y_val'], cwd,
                                    n_trials=optuna_trials, sw_tr=d['w_train'], sw_va=d['w_val'])
        pred_te = lgb_mdl.predict(d['X_test'])
        pred_va = lgb_mdl.predict(d['X_val'])
        all_models[ticker]['LightGBM'] = lgb_mdl
        all_results[ticker]['LightGBM'] = {
            'acc': accuracy_score(d['y_test'], pred_te),
            'f1': f1_score(d['y_test'], pred_te, average='macro', sample_weight=d['w_test']),
            'val_f1': f1_score(d['y_val'], pred_va, average='macro', sample_weight=d['w_val']),
            'optuna_f1': lgb_f1,
        }
        print(f'    LightGBM acc={all_results[ticker]["LightGBM"]["acc"]:.4f} f1={all_results[ticker]["LightGBM"]["f1"]:.4f} val_f1={all_results[ticker]["LightGBM"]["val_f1"]:.4f}')

        xgb_mdl, xgb_f1 = tune_xgb(d['X_train'], d['y_train'], d['X_val'], d['y_val'], cwd,
                                    n_trials=optuna_trials, sw_tr=d['w_train'], sw_va=d['w_val'])
        pred_te = xgb_mdl.predict(d['X_test'])
        pred_va = xgb_mdl.predict(d['X_val'])
        all_models[ticker]['XGBoost'] = xgb_mdl
        all_results[ticker]['XGBoost'] = {
            'acc': accuracy_score(d['y_test'], pred_te),
            'f1': f1_score(d['y_test'], pred_te, average='macro', sample_weight=d['w_test']),
            'val_f1': f1_score(d['y_val'], pred_va, average='macro', sample_weight=d['w_val']),
            'optuna_f1': xgb_f1,
        }
        print(f'    XGBoost  acc={all_results[ticker]["XGBoost"]["acc"]:.4f} f1={all_results[ticker]["XGBoost"]["f1"]:.4f} val_f1={all_results[ticker]["XGBoost"]["val_f1"]:.4f}')

        for model_name, build_fn in [('GRU', build_gru), ('LSTM', build_lstm), ('Transformer', build_transformer)]:
            with tf.device(TF_DEVICE):
                mdl = build_fn()
                mdl.fit(d['Xtr_seq'], d['ytr_seq'],
                        validation_data=(d['Xva_seq'], d['yva_seq'], d['wva_seq']),
                        sample_weight=d['wtr_seq'],
                        epochs=dl_epochs, batch_size=BATCH,
                        callbacks=get_dl_callbacks(total_epochs=dl_epochs, patience=14),
                        verbose=0)
            pred_te = mdl.predict(d['Xte_seq'], verbose=0).argmax(axis=1)
            pred_va = mdl.predict(d['Xva_seq'], verbose=0).argmax(axis=1)
            all_models[ticker][model_name] = mdl
            all_results[ticker][model_name] = {
                'acc': accuracy_score(d['yte_seq'], pred_te),
                'f1': f1_score(d['yte_seq'], pred_te, average='macro', sample_weight=d['wte_seq']),
                'val_f1': f1_score(d['yva_seq'], pred_va, average='macro', sample_weight=d['wva_seq']),
            }
            print(f'    {model_name:11s} acc={all_results[ticker][model_name]["acc"]:.4f} f1={all_results[ticker][model_name]["f1"]:.4f} val_f1={all_results[ticker][model_name]["val_f1"]:.4f}')
    print('Training complete')
    return all_models, all_results

all_models, all_results = train_all_models()


In [ ]:
# Ensemble weighting + validation-tuned decision rules

from scipy.optimize import minimize
ens_weights = {}
FINAL_MODELS = {}
decision_rules = {}
model_decision_rules = {}

def _as_proba(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, dtype=np.float64)
    if p.ndim == 1:
        p = p.reshape(1, -1)
    p = np.nan_to_num(p, nan=1/3, posinf=1/3, neginf=1/3)
    p = np.clip(p, 1e-8, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def _align_proba(p: np.ndarray, n: int) -> np.ndarray:
    p = _as_proba(p)
    if len(p) < n:
        p = np.vstack([np.full((n - len(p), 3), [0, 1, 0]), p])
    return p[-n:]

def get_proba(ticker: str, model_name: str, X_flat: np.ndarray, X_seq: np.ndarray) -> np.ndarray:
    mdl = all_models[ticker][model_name]
    return _as_proba(mdl.predict(X_seq, verbose=0)) if model_name in DL_NAMES else _as_proba(mdl.predict_proba(X_flat))

def _default_rule() -> dict:
    return {'buy_conf': MODEL_CONF_MIN, 'sell_conf': MODEL_CONF_MIN, 'edge': PROB_THRESHOLD, 'flat_cap': 0.65}

def signals_from_proba(ticker: str, probs: np.ndarray, rule: dict = None) -> np.ndarray:
    p = _as_proba(probs)
    r = rule or decision_rules.get(ticker, _default_rule())
    pred = np.ones(len(p), dtype=np.int32)
    up_edge = p[:, 2] - p[:, 0]
    dn_edge = p[:, 0] - p[:, 2]
    buy = (p[:, 2] >= r['buy_conf']) & (up_edge >= r['edge']) & (p[:, 1] <= r['flat_cap'])
    sell = (p[:, 0] >= r['sell_conf']) & (dn_edge >= r['edge']) & (p[:, 1] <= r['flat_cap'])
    pred[buy] = 2
    pred[sell] = 0
    return pred

def tune_decision_rule(y_true: np.ndarray, probs: np.ndarray, sample_weight: np.ndarray = None) -> dict:
    probs = _as_proba(probs)
    y_true = np.asarray(y_true, dtype=np.int32)
    best_rule = _default_rule()
    best_score = -1e9
    conf_grid = [0.34, 0.38, 0.42, 0.46, 0.50, 0.55, 0.60, 0.66]
    edge_grid = [0.02, 0.04, 0.06, 0.08, 0.10, 0.13, 0.16, 0.20]
    flat_grid = [0.48, 0.55, 0.62, 0.70, 0.80]
    for conf in conf_grid:
        for edge in edge_grid:
            for flat_cap in flat_grid:
                rule = {'buy_conf': conf, 'sell_conf': conf, 'edge': edge, 'flat_cap': flat_cap}
                pred = signals_from_proba('', probs, rule)
                trade_rate = float(np.mean(pred != 1))
                f1 = f1_score(y_true, pred, average='macro', sample_weight=sample_weight)
                penalty = 0.02 * max(0.0, trade_rate - 0.60) + 0.01 * max(0.0, 0.03 - trade_rate)
                score = f1 - penalty
                if score > best_score:
                    best_score = score
                    best_rule = rule
    best_rule['score'] = float(best_score)
    return best_rule

def ensemble_predict(ticker: str, X_seq: np.ndarray, X_flat: np.ndarray, weights: np.ndarray = None, use_rule: bool = True) -> tuple:
    model_names = list(all_models[ticker].keys())
    if weights is None:
        weights = ens_weights.get(ticker, np.ones(len(model_names)) / len(model_names))
    probs = np.zeros((len(X_seq), 3), dtype=np.float64)
    for w, mn in zip(weights, model_names):
        p = _align_proba(get_proba(ticker, mn, X_flat, X_seq), len(X_seq))
        probs += float(w) * p
    probs = _as_proba(probs)
    pred = signals_from_proba(ticker, probs) if use_rule else probs.argmax(axis=1).astype(np.int32)
    return pred, probs

def rebuild_ensembles():
    global ens_weights, FINAL_MODELS, decision_rules, model_decision_rules
    ens_weights, FINAL_MODELS, decision_rules, model_decision_rules = {}, {}, {}, {}
    for name, ticker in TICKERS.items():
        d = datasets[ticker]
        mns = list(all_models[ticker].keys())
        n_va, n_te = len(d['yva_seq']), len(d['yte_seq'])
        proba_val = []
        proba_test = {}
        candidate_rules = {}
        candidate_scores = {}

        for mn in mns:
            pv = _align_proba(get_proba(ticker, mn, d['X_val'], d['Xva_seq']), n_va)
            pt = _align_proba(get_proba(ticker, mn, d['X_test'], d['Xte_seq']), n_te)
            proba_val.append(pv)
            proba_test[mn] = pt
            rule = tune_decision_rule(d['yva_seq'], pv, d['wva_seq'])
            pred_va = signals_from_proba(ticker, pv, rule)
            pred_te = signals_from_proba(ticker, pt, rule)
            val_f1 = f1_score(d['yva_seq'], pred_va, average='macro', sample_weight=d['wva_seq'])
            test_f1 = f1_score(d['yte_seq'], pred_te, average='macro', sample_weight=d['wte_seq'])
            all_results[ticker][mn]['val_f1_rule'] = val_f1
            all_results[ticker][mn]['f1_rule'] = test_f1
            candidate_rules[mn] = rule
            candidate_scores[mn] = val_f1

        def neg_f1(w):
            w = np.abs(w); w /= w.sum() + 1e-9
            p = _as_proba(sum(wi * pi for wi, pi in zip(w, proba_val)))
            pred = p.argmax(axis=1)
            return -f1_score(d['yva_seq'], pred, average='macro', sample_weight=d['wva_seq'])

        init_w = np.ones(len(mns)) / len(mns)
        res = minimize(neg_f1, init_w, method='SLSQP', bounds=[(0, 1)] * len(mns),
                       constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}, options={'maxiter': 500})
        w_opt = np.abs(res.x); w_opt /= w_opt.sum() + 1e-9
        ens_weights[ticker] = w_opt

        ens_val = _as_proba(sum(wi * pi for wi, pi in zip(w_opt, proba_val)))
        ens_rule = tune_decision_rule(d['yva_seq'], ens_val, d['wva_seq'])
        ens_pred_va = signals_from_proba(ticker, ens_val, ens_rule)
        ens_val_f1 = f1_score(d['yva_seq'], ens_pred_va, average='macro', sample_weight=d['wva_seq'])
        _, ens_test = ensemble_predict(ticker, d['Xte_seq'], d['X_test'], w_opt, use_rule=False)
        ens_pred_te = signals_from_proba(ticker, ens_test, ens_rule)
        all_results[ticker]['Ensemble'] = {
            'acc': accuracy_score(d['yte_seq'], ens_pred_te),
            'f1': f1_score(d['yte_seq'], ens_pred_te, average='macro', sample_weight=d['wte_seq']),
            'val_f1': ens_val_f1,
            'val_f1_rule': ens_val_f1,
            'f1_rule': f1_score(d['yte_seq'], ens_pred_te, average='macro', sample_weight=d['wte_seq']),
        }
        candidate_rules['Ensemble'] = ens_rule
        candidate_scores['Ensemble'] = ens_val_f1

        best_name = max(candidate_scores, key=lambda k: candidate_scores[k])
        FINAL_MODELS[ticker] = best_name
        decision_rules[ticker] = candidate_rules[best_name]
        model_decision_rules[ticker] = candidate_rules
        r = decision_rules[ticker]
        print(f'  {name}: final={best_name} val_f1={candidate_scores[best_name]:.4f} test_f1={all_results[ticker][best_name].get("f1_rule", all_results[ticker][best_name].get("f1",0)):.4f} rule=conf:{r["buy_conf"]:.2f} edge:{r["edge"]:.2f} flat:{r["flat_cap"]:.2f}')
    return ens_weights, FINAL_MODELS

ens_weights, FINAL_MODELS = rebuild_ensembles()


In [ ]:
# ════════════════════════════════════════════════════════════════
#  RiskManager v4
#  ★ 버그 수정: 손절 최소 보유봉 분리 (STOP_MIN_BARS)
# ════════════════════════════════════════════════════════════════

class RiskManager:
    def __init__(self,
                 initial_capital = INITIAL_CAPITAL,
                 stop_loss       = STOP_LOSS_PCT,
                 take_profit     = TAKE_PROFIT_PCT,
                 daily_limit     = DAILY_LOSS_LIMIT,
                 kelly_frac      = KELLY_FRACTION,
                 slippage        = SLIPPAGE_RATE,
                 fee             = FEE_RATE):
        self.initial_capital = initial_capital
        self.stop_loss       = stop_loss
        self.take_profit     = take_profit
        self.daily_limit     = daily_limit
        self.kelly_frac      = kelly_frac
        self.slippage        = slippage
        self.fee             = fee
        self.daily_start_eq  = initial_capital
        self.trading_halted  = False
        self.positions       = {}
        self.last_reset      = datetime.date.today()

    def reset_daily(self, equity: float, date=None):
        today = date or datetime.date.today()
        if today != self.last_reset:
            self.daily_start_eq = equity
            self.trading_halted = False
            self.last_reset     = today

    def check_daily_limit(self, equity: float) -> bool:
        loss_ratio = (equity - self.daily_start_eq) / (self.daily_start_eq + 1e-9)
        if loss_ratio < -self.daily_limit:
            self.trading_halted = True
        return self.trading_halted

    def check_stop_take(self, ticker: str, price: float,
                        hold_bar: int = 999) -> str:
        """
        ★ v4 수정: hold_bar < STOP_MIN_BARS 이면 손절/익절 체크 안 함.
        (손절도 최소 3봉 뒤부터 허용)
        """
        if ticker not in self.positions:
            return 'hold'
        if hold_bar < STOP_MIN_BARS:
            return 'hold'
        entry = self.positions[ticker]['entry_price']
        ret   = (price - entry) / (entry + 1e-9)
        if ret <= -self.stop_loss:  return 'stop_loss'
        if ret >= self.take_profit: return 'take_profit'
        return 'hold'

    def open_position(self, ticker: str, entry_price: float, qty: float):
        if ticker not in self.positions:
            self.positions[ticker] = {
                'entry_price': entry_price,
                'qty':         qty,
            }

    def close_position(self, ticker: str):
        self.positions.pop(ticker, None)

    def kelly_size(self, cash: float, win_rate: float,
                   avg_win: float, avg_loss: float,
                   n_trades: int) -> float:
        if n_trades < KELLY_MIN_TRADES:
            return cash * KELLY_FIXED_FRAC
        if avg_loss <= 0 or avg_win <= 0:
            return cash * KELLY_FIXED_FRAC
        b     = avg_win / avg_loss
        kelly = (b * win_rate - (1 - win_rate)) / b
        if kelly <= 0:
            return 0.0
        kelly = min(kelly, 1.0)
        return cash * kelly * self.kelly_frac

    def apply_slippage(self, price: float, is_buy: bool) -> float:
        return price * (1 + self.slippage) if is_buy else price * (1 - self.slippage)

    def signal_confidence(self, prob: np.ndarray) -> bool:
        """
        ★ v4 강화: 횡보 확률이 가장 높으면 무조건 홀드
        """
        max_prob = float(max(prob))
        if max_prob < MODEL_CONF_MIN:
            return False
        if np.argmax(prob) == 1:          # 횡보 신호 최대
            return False
        bull_bear_diff = abs(float(prob[2]) - float(prob[0]))
        if bull_bear_diff < PROB_THRESHOLD:
            return False
        return True


print('✅ RiskManager v4 준비 완료')
print(f'  손절: {STOP_LOSS_PCT*100:.0f}%  익절: {TAKE_PROFIT_PCT*100:.0f}%'
      f'  일일한도: {DAILY_LOSS_LIMIT*100:.0f}%')
print(f'  손절 최소 보유: {STOP_MIN_BARS}봉  신호 매도 최소: {MIN_HOLD_BARS}봉')


In [ ]:
# ════════════════════════════════════════════════════════════════
#  백테스트 v4
#  ★ 버그 수정:
#    1. exec_price 경계: 마지막 봉 skip
#    2. hold_bar → RiskManager로 단일 관리
#    3. 피라미딩 코드 제거 (coin==0 조건과 일관성 유지)
#    4. check_stop_take에 hold_bar 전달
# ════════════════════════════════════════════════════════════════

def backtest_full(ticker: str, initial: float = INITIAL_CAPITAL) -> dict:
    rm   = RiskManager(initial_capital=initial)
    d    = datasets[ticker]
    df_t = d['test_df'].copy()
    fm   = FINAL_MODELS[ticker]

    # ── 예측값 준비 ──
    if fm in DL_NAMES:
        probs  = all_models[ticker][fm].predict(d['Xte_seq'], verbose=0)
        yp_cls = probs.argmax(axis=1)
        pad    = len(df_t) - len(yp_cls)
        yp_all   = np.concatenate([np.ones(pad, dtype=int), yp_cls])
        prob_all = np.concatenate([np.full((pad,3),[0,1,0]), probs])
    elif fm == 'Ensemble':
        yp_cls, probs = ensemble_predict(ticker, d['Xte_seq'], d['X_test'])
        n_te   = len(d['yte_seq'])
        yp_cls = yp_cls[-n_te:]
        probs  = probs[-n_te:]
        pad    = len(df_t) - len(yp_cls)
        yp_all   = np.concatenate([np.ones(pad, dtype=int), yp_cls])
        prob_all = np.concatenate([np.full((pad,3),[0,1,0]), probs])
    else:
        yp_cls   = all_models[ticker][fm].predict(d['X_test'])
        prob_all = all_models[ticker][fm].predict_proba(d['X_test'])
        yp_all   = yp_cls

    yp_all = signals_from_proba(ticker, prob_all, decision_rules.get(ticker, None))

    prices = df_t['close'].values
    opens  = df_t['open'].values
    dates  = df_t.index
    n      = len(prices)

    cash, coin, entry = initial, 0.0, None
    hold_bar   = 0
    trade_rets = []
    equity_curve = []
    trade_cnt  = {'매수':0,'매도':0,'손절':0,'익절':0,'홀드_필터':0}

    for i in range(n - 1):      # ★ 마지막 봉 skip (다음봉 open 없음)
        p          = prices[i]
        exec_price = opens[i + 1]   # ★ 다음봉 open 체결
        sig        = yp_all[i]
        prob       = prob_all[i]

        cur_equity = cash + coin * p
        rm.reset_daily(cur_equity, dates[i].date())
        if rm.check_daily_limit(cur_equity):
            equity_curve.append({'datetime':dates[i],'price':p,'cash':cash,
                                  'coin':coin,'equity':cur_equity,'action':'중단'})
            if coin > 0: hold_bar += 1
            continue

        action = '홀드'

        # ── 손절 / 익절 체크 ──
        if coin > 0 and entry is not None:
            chk = rm.check_stop_take(ticker, exec_price, hold_bar)
            if chk in ('stop_loss', 'take_profit'):
                sp    = rm.apply_slippage(exec_price, is_buy=False)
                cash += coin * sp * (1 - FEE_RATE)
                trade_rets.append((sp - entry) / entry)
                rm.close_position(ticker)
                action   = '손절' if chk == 'stop_loss' else '익절'
                trade_cnt[action] += 1
                coin, entry, hold_bar = 0.0, None, 0

        confident = rm.signal_confidence(prob, ticker)

        # ── 매수 ──
        if sig == 2 and cash > MIN_ORDER and action == '홀드' and coin == 0:
            if not confident:
                trade_cnt['홀드_필터'] += 1
            else:
                wins   = [r for r in trade_rets if r > 0]
                losses = [abs(r) for r in trade_rets if r <= 0]
                n_tr   = len(trade_rets)
                wr   = len(wins) / n_tr if n_tr > 0 else 0.5
                aw   = np.mean(wins)   if wins   else TAKE_PROFIT_PCT
                al   = np.mean(losses) if losses else STOP_LOSS_PCT
                amt  = rm.kelly_size(cash, wr, aw, al, n_tr)
                if amt <= 0:
                    trade_cnt['홀드_필터'] += 1
                else:
                    amt  = min(amt, cash * 0.95)
                    bp   = rm.apply_slippage(exec_price, is_buy=True)
                    qty  = amt * (1 - FEE_RATE) / bp
                    rm.open_position(ticker, bp, qty)
                    coin, cash, entry, hold_bar = qty, cash - amt, bp, 0
                    action = '매수'
                    trade_cnt['매수'] += 1

        # ── 매도 ──
        elif sig == 0 and coin > 0 and action == '홀드':
            if hold_bar < MIN_HOLD_BARS:
                pass   # 최소 보유 미달
            elif not confident:
                trade_cnt['홀드_필터'] += 1
            else:
                sp    = rm.apply_slippage(exec_price, is_buy=False)
                cash += coin * sp * (1 - FEE_RATE)
                if entry is not None:
                    trade_rets.append((sp - entry) / entry)
                rm.close_position(ticker)
                coin, entry, hold_bar = 0.0, None, 0
                action = '매도'
                trade_cnt['매도'] += 1

        if coin > 0:
            hold_bar += 1

        equity_curve.append({'datetime':dates[i],'price':p,'cash':cash,
                              'coin':coin,'equity':cash+coin*p,'action':action,
                              'signal':int(sig)})

    # ── 미청산 포지션 마지막 가격으로 평가 ──
    final_equity = cash + coin * prices[-1]
    eq_df  = pd.DataFrame(equity_curve).set_index('datetime')
    eq_df.loc[dates[-1]] = {
        'price':prices[-1],'cash':cash,'coin':coin,
        'equity':final_equity,'action':'종료','signal':1
    }
    fe   = eq_df['equity'].iloc[-1]
    roll = eq_df['equity'].cummax()
    dd   = (eq_df['equity'] - roll) / (roll + 1e-9) * 100
    dr   = eq_df['equity'].pct_change().dropna()
    wins = [r for r in trade_rets if r > 0]

    print(f'  [{ticker}] 매수:{trade_cnt["매수"]}  매도:{trade_cnt["매도"]}  '
          f'손절:{trade_cnt["손절"]}  익절:{trade_cnt["익절"]}  필터:{trade_cnt["홀드_필터"]}')

    return {
        'ticker':       ticker,
        'final_model':  fm,
        'initial':      initial,
        'final_equity': fe,
        'total_return': (fe - initial) / initial * 100,
        'bah_return':   (prices[-1] - prices[0]) / prices[0] * 100,
        'max_drawdown': dd.min(),
        'sharpe':       dr.mean() / (dr.std() + 1e-9) * np.sqrt(BARS_PER_DAY * 365),
        'win_rate':     len(wins) / len(trade_rets) * 100 if trade_rets else 0,
        'n_trades':     len(trade_rets),
        'trade_cnt':    trade_cnt,
        'equity_df':    eq_df,
    }


bt_results = {}
print('=' * 72)
print('  Backtest v4 (5m base + 15m/1h features | paper simulation)')
print('=' * 72)
print(f'{"코인":6s}  {"최종자산":>13s}  {"수익률":>9s}  {"BaH":>9s}  '
      f'{"MDD":>8s}  {"샤프":>7s}  {"승률":>6s}  {"거래":>5s}')
print('-' * 72)
for name, ticker in TICKERS.items():
    r = backtest_full(ticker)
    bt_results[ticker] = r
    s = '+' if r['total_return'] >= 0 else ''
    print(f'  {name:6s}  {r["final_equity"]:>13,.0f}원  '
          f'{s}{r["total_return"]:>8.2f}%  {r["bah_return"]:>9.2f}%  '
          f'{r["max_drawdown"]:>7.2f}%  {r["sharpe"]:>7.3f}  '
          f'{r["win_rate"]:>5.1f}%  {r["n_trades"]:>5d}')


In [ ]:
# ????????????????????????????????????????????????????????????????
#  AutoTraderV3 (paper-only continuous collector/trainer)
# ????????????????????????????????????????????????????????????????

class TelegramNotifier:
    def send(self, text: str):
        return
    def notify_trade(self, ticker, action, price, amount, pnl, equity):
        return

class Journal:
    def __init__(self, path='logs/paper_trades.csv'):
        self.path = path
        self.rows = []
    def log(self, row: dict):
        self.rows.append(row)
        pd.DataFrame(self.rows).to_csv(self.path, index=False, encoding='utf-8-sig')
    def recent(self, n=10) -> pd.DataFrame:
        return pd.DataFrame(self.rows).tail(n) if self.rows else pd.DataFrame()

class AutoTraderV3:
    LOOP_SEC = 300
    def __init__(self, upbit_obj, tg: TelegramNotifier, jnl: Journal):
        self.upbit = None
        self.tg = tg
        self.jnl = jnl
        self.risk = RiskManager()
        self.entries = {}
        self.hold_bars = {}
        self.hist = []
        self.paper_bal = {'KRW': INITIAL_CAPITAL}
        self.cycle = 0
        self.prob_ema = {}

    def _refresh_ticker_data(self, ticker: str):
        raw_data[ticker] = {}
        for tf_name in TIMEFRAMES:
            raw_data[ticker][tf_name] = update_ohlcv_cache(ticker, tf_name, full=False)

    def _get_data(self, ticker: str) -> pd.DataFrame:
        self._refresh_ticker_data(ticker)
        df = add_features(raw_data[ticker])
        return df if len(df) >= SEQUENCE_LEN else pd.DataFrame()

    def _continuous_retrain_if_needed(self):
        if not CONTINUOUS_LEARNING or self.cycle == 0 or self.cycle % RETRAIN_EVERY_CYCLES != 0:
            return
        print('\n  Retraining with accumulated live data...')
        prepare_datasets()
        train_all_models(optuna_trials=FAST_OPTUNA_TRIALS, dl_epochs=max(20, EPOCHS // 2))
        rebuild_ensembles()
        print('  Retraining complete. Live model set swapped.')

    def _predict(self, ticker: str, df: pd.DataFrame):
        sc = datasets[ticker]['scaler']
        feats = [f for f in FEATURE_COLS if f in df.columns]
        X_raw = df[feats].tail(SEQUENCE_LEN + 1).values
        X_s = sc.transform(X_raw)
        fm = FINAL_MODELS[ticker]
        if fm in DL_NAMES:
            X_seq = X_s[-SEQUENCE_LEN:].reshape(1, SEQUENCE_LEN, len(feats))
            prob = all_models[ticker][fm].predict(X_seq, verbose=0)[0]
        elif fm == 'Ensemble':
            X_seq = X_s[-SEQUENCE_LEN:].reshape(1, SEQUENCE_LEN, len(feats))
            _, ps = ensemble_predict(ticker, X_seq, X_s[-1:])
            prob = ps[0]
        else:
            prob = all_models[ticker][fm].predict_proba(X_s[-1:])[0]
        prob = np.asarray(prob, dtype=np.float64)
        prev = self.prob_ema.get(ticker)
        if prev is not None:
            prob = SIGNAL_EMA_ALPHA * prob + (1 - SIGNAL_EMA_ALPHA) * prev
        prob = prob / (prob.sum() + 1e-9)
        self.prob_ema[ticker] = prob
        sig = int(signals_from_proba(ticker, prob.reshape(1, -1), decision_rules.get(ticker, None))[0])
        return sig, prob

    def _bal(self, cur: str) -> float:
        return float(self.paper_bal.get(cur, 0.0))

    def _trade(self, ticker: str, sig: int, prob: np.ndarray, price: float):
        cur = ticker.split('-')[1]
        krw = self._bal('KRW')
        coin = self._bal(cur)
        equity = krw + coin * price
        self.risk.reset_daily(equity)
        if self.risk.check_daily_limit(equity):
            print(f'  STOP {ticker}: daily paper-loss limit hit')
            return
        action, pnl = 'HOLD', None
        hb = self.hold_bars.get(ticker, 0)
        if coin * price > MIN_ORDER and ticker in self.entries:
            chk = self.risk.check_stop_take(ticker, price, hb)
            if chk in ('stop_loss', 'take_profit'):
                self.paper_bal['KRW'] = self.paper_bal.get('KRW', 0.0) + coin * price * (1 - FEE_RATE)
                self.paper_bal[cur] = 0.0
                entry = self.entries.pop(ticker, price)
                pnl = (price - entry) / entry
                self.hist.append(pnl)
                self.risk.close_position(ticker)
                self.hold_bars[ticker] = 0
                action = 'STOP' if chk == 'stop_loss' else 'TAKE'
        confident = self.risk.signal_confidence(prob, ticker)
        if sig == 2 and krw > MIN_ORDER and action == 'HOLD' and coin * price < MIN_ORDER and confident:
            wins = [r for r in self.hist if r > 0]
            losses = [abs(r) for r in self.hist if r <= 0]
            n_tr = len(self.hist)
            wr = len(wins) / n_tr if n_tr > 0 else 0.5
            aw = np.mean(wins) if wins else TAKE_PROFIT_PCT
            al = np.mean(losses) if losses else STOP_LOSS_PCT
            amt = min(self.risk.kelly_size(krw, wr, aw, al, n_tr), krw * 0.95)
            if amt > MIN_ORDER:
                self.paper_bal['KRW'] = max(0.0, krw - amt)
                self.paper_bal[cur] = self.paper_bal.get(cur, 0.0) + (amt * (1 - FEE_RATE) / price)
                self.entries[ticker] = price
                self.hold_bars[ticker] = 0
                self.risk.open_position(ticker, price, amt / price)
                action = 'BUY'
        elif sig == 0 and coin * price > MIN_ORDER and action == 'HOLD' and hb >= MIN_HOLD_BARS and confident:
            self.paper_bal['KRW'] = self.paper_bal.get('KRW', 0.0) + coin * price * (1 - FEE_RATE)
            self.paper_bal[cur] = 0.0
            entry = self.entries.pop(ticker, price)
            pnl = (price - entry) / entry
            self.hist.append(pnl)
            self.risk.close_position(ticker)
            self.hold_bars[ticker] = 0
            action = 'SELL'
        if self._bal(cur) * price > MIN_ORDER and action == 'HOLD':
            self.hold_bars[ticker] = hb + 1
        if action != 'HOLD':
            equity = self.paper_bal.get('KRW', 0.0) + self.paper_bal.get(cur, 0.0) * price
            self.jnl.log({'timestamp': datetime.datetime.now().isoformat(), 'ticker': ticker, 'action': action, 'price': price, 'pnl_pct': round(pnl * 100, 4) if pnl is not None else None, 'total_equity': round(equity, 0), 'signal': sig, 'prob_down': round(float(prob[0]), 4), 'prob_neutral': round(float(prob[1]), 4), 'prob_up': round(float(prob[2]), 4), 'model_used': FINAL_MODELS[ticker], 'hold_bars': hb})
        print(f'  {datetime.datetime.now().strftime("%m/%d %H:%M")} {ticker:10s} sig={sig} price={price:,.0f} action={action} equity={self.paper_bal.get("KRW",0)+self.paper_bal.get(cur,0)*price:,.0f}')

    def run(self, tickers, max_iter=None):
        print('=' * 60)
        print('  Paper trader: 5m base + 15m/1h features + continuous learning')
        itr = 0
        try:
            while True:
                if max_iter and itr >= max_iter:
                    break
                itr += 1
                self.cycle = itr
                print(f'\n  --- cycle #{itr:04d} {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")} ---')
                for ticker in tickers:
                    try:
                        df = self._get_data(ticker)
                        if df.empty:
                            print(f'  {ticker}: no data')
                            continue
                        price = float(df['close'].iloc[-1])
                        sig, prob = self._predict(ticker, df)
                        self._trade(ticker, sig, prob, price)
                        print(f'    prob down={prob[0]:.3f} flat={prob[1]:.3f} up={prob[2]:.3f}')
                    except Exception as e:
                        print(f'  {ticker} error: {e}')
                    time.sleep(0.5)
                self._continuous_retrain_if_needed()
                time.sleep(self.LOOP_SEC)
        except KeyboardInterrupt:
            print('Paper trader stopped')
        recent = self.jnl.recent(10)
        if not recent.empty:
            print(recent.to_string(index=False))

telegram = TelegramNotifier()
journal = Journal()
trader = AutoTraderV3(None, telegram, journal)
print('Paper AutoTrader initialized')
print('  continuous paper run: trader.run(list(TICKERS.values()))')
print('  smoke test: trader.run(list(TICKERS.values()), max_iter=3)')


In [ ]:
# ════════════════════════════════════════════════════════════════
#  모델 저장 + 최종 요약
# ════════════════════════════════════════════════════════════════

print('💾 모델 저장 중...')
for name, ticker in TICKERS.items():
    fm  = FINAL_MODELS[ticker]
    mdl = all_models[ticker].get(fm)
    st  = ticker.replace('-', '_')
    if mdl and fm in DL_NAMES:
        path = f'models/final/{st}_{fm}.keras'
        mdl.save(path)
    elif mdl:
        path = f'models/final/{st}_{fm}.pkl'
        joblib.dump(mdl, path)
    else:
        path = '(Ensemble - individual models saved)'
    joblib.dump(datasets[ticker]['scaler'], f'models/final/{st}_scaler.pkl')
    if ticker in ens_weights:
        np.save(f'models/final/{st}_ens_weights.npy', ens_weights[ticker])
    print(f'  ✅ {name:8s} → {path}')

# config 저장
cfg = {
    'TICKERS':         TICKERS,
    'TIMEFRAMES':      TIMEFRAMES,
    'BASE_TIMEFRAME':  BASE_TIMEFRAME,
    'GPU_ONLY':        GPU_ONLY,
    'PAPER_TRADING_ONLY': PAPER_TRADING_ONLY,
    'CONTINUOUS_LEARNING': CONTINUOUS_LEARNING,
    'FEATURE_COLS':    FEATURE_COLS,
    'FINAL_MODELS':    FINAL_MODELS,
    'DECISION_RULES':   decision_rules,
    'CANDLE_INTERVAL': CANDLE_INTERVAL,
    'SEQUENCE_LEN':    SEQUENCE_LEN,
    'PREDICT_HORIZON': PREDICT_HORIZON,
    'LABEL_THRESHOLD': LABEL_THRESHOLD,
    'STOP_LOSS_PCT':   STOP_LOSS_PCT,
    'TAKE_PROFIT_PCT': TAKE_PROFIT_PCT,
    'DAILY_LOSS_LIMIT':DAILY_LOSS_LIMIT,
    'KELLY_FRACTION':  KELLY_FRACTION,
    'MIN_HOLD_BARS':   MIN_HOLD_BARS,
    'STOP_MIN_BARS':   STOP_MIN_BARS,
    'MODEL_CONF_MIN':  MODEL_CONF_MIN,
    'PROB_THRESHOLD':  PROB_THRESHOLD,
    'version':         'v4.0',
    'saved_at':        datetime.datetime.now().isoformat(),
}
with open('models/final/config.json', 'w', encoding='utf-8') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

# ── 최종 요약 출력 ──
W = 72
print()
print('╔' + '═'*W + '╗')
print('║' + '  🏆 업비트 자동 매매 시스템 v4.0 — 최종 요약'.center(W) + '║')
print('╠' + '═'*W + '╣')
print(f'║  {"코인":8s}  {"최종모델":16s}  {"정확도":>8s}  {"F1":>8s}  {"WF_F1":>8s}  ║')
print('╠' + '═'*W + '╣')
for name, ticker in TICKERS.items():
    fm  = FINAL_MODELS[ticker]
    m   = all_results[ticker].get(fm, {})
    wf  = wf_results.get(ticker, {}).get('mean_f1', 0)
    print(f'║  {name:8s}  {fm:16s}  {m.get("acc",0):8.4f}  '
          f'{m.get("f1",0):8.4f}  {wf:8.4f}  ║')
print('╠' + '═'*W + '╣')
print(f'║  {"코인":8s}  {"최종자산":>13s}  {"수익률":>9s}  '
      f'{"BaH":>9s}  {"MDD":>7s}  {"샤프":>6s}  ║')
print('╠' + '═'*W + '╣')
for name, ticker in TICKERS.items():
    r = bt_results[ticker]
    s = '+' if r['total_return'] >= 0 else ''
    print(f'║  {name:8s}  {r["final_equity"]:>13,.0f}원  '
          f'{s}{r["total_return"]:>8.2f}%  '
          f'{r["bah_return"]:>9.2f}%  '
          f'{r["max_drawdown"]:>6.2f}%  '
          f'{r["sharpe"]:>6.3f}  ║')
print('╠' + '═'*W + '╣')
print('║  ✅ v4 핵심 개선:' + ' '*54 + '║')
print('\u2551   5m base + 15m/1h features | Horizon12 | label +/-0.6% | paper only \u2551')
print('\u2551   Leakage control | WF | dynamic thresholds | recency weighting      \u2551')
print('║   exec_price 경계 수정 | 횡보신호 필터 | V3 클래스명 통일       ║')
print('╚' + '═'*W + '╝')
print()
print('  paper continuous run: trader.run(list(TICKERS.values()))')
